In [2]:
import cv2
import glob
import os
import numpy as np
from tqdm import tqdm

ModuleNotFoundError: No module named 'cv2'

### Frame class

Setting up the frame class for ease of access to the images and their respective extrinsic information

In [ ]:
class Frame:
    def __init__(self, id, img):
        self.id = id
        self.img = img
        # Keypoint info
        self.kp = None   
        self.des = None
        # Extrinsic info
        self.R = None
        self.t = None
        self.registered = False

### Image loading

We load up the Ammonite image dataset, consisting of 102 images of an Ammonite fossil at different angles and heights

In [ ]:
# Load all images from the Ammonite folder 
folder = "Ammonite"
image_paths = sorted(glob.glob(os.path.join(folder, "*.jpg")))
frames = []

for path in tqdm(image_paths, desc="Images loaded"):
    frames.append(Frame(path, cv2.imread(path)))

Images loaded: 100%|██████████| 102/102 [00:16<00:00,  6.08it/s]


### Camera parameter setup

From the image metadata, we have that:
- The camera is a Sony Alpha 77, with a sensor size of 23.5 mm x 15.6 mm
- The focal length for all images is set at 50 mm.
- The images themselves are 6000 x 4000 pixels

However, 6000 x 4000 can make feature matching calcuations take an excessive time to finish, and SIFT can work well with smaller image data. We will convert our images to 1500 x 1000 instead

In [ ]:
for frame in tqdm(frames, desc="Resizing"):
    frame.img = cv2.resize(frame.img, (1500, 1000))

Resizing: 100%|██████████| 102/102 [00:00<00:00, 121.04it/s]


We can now calculate the extrinsic matrix as such:

In [ ]:
# Basic parameters
foc = 50
img_width = 1000
img_height = 1500
sens_width = 23.5
sens_length = 15.6

# Intrinsic parameters
fx = (foc / sens_width) * img_width
fy = (foc / sens_length) * img_width
cx = img_width / 2
cy = img_height / 2

# Construct intrinsic matrix
intrin = np.array([[fx, 0, cx],
                   [0, fy, cy],
                   [0,  0,  1]])

print(intrin)

[[2.12765957e+03 0.00000000e+00 5.00000000e+02]
 [0.00000000e+00 3.20512821e+03 7.50000000e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]


### Feature detection

We will use SIFT in this example, since we have plenty of images of our object and it is computionally cheap.

In [ ]:
# We are using SIFT in this example
sift = cv2.SIFT_create()

# Use the SIFT algorithm to pickout keypoints and their descriptors
for frame in tqdm(frames):
    gray = cv2.cvtColor(frame.img, cv2.COLOR_BGR2GRAY)
    kp, des = sift.detectAndCompute(gray, None)
    frame.kp = kp
    frame.des = des

100%|██████████| 102/102 [00:16<00:00,  6.01it/s]


### Feature matching

Our feature matching strategy is as follows:

- The images, when looked at sequentially, trace out a circular path around the object at different heights. Therefore, the order of the images follow a chronology.
- Instead pairing up all (102 x 101) pair of images and feature matching on them, we can feature match across successive pairs of images to give

In [ ]:
# This function uses FLANN + Lowe ratio test to feature match
def match_pair(img1, img2, ratio=0.75):
    # FLANN parameters for SIFT
    index_params = {'algorithm': 1, 'trees': 5}
    search_params = {'algorithm': 1, 'trees': 5}
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    # Create and compare matches, by finding 2 nearest neighbors of each match, 
    # and then applying the ratio test
    matches = flann.knnMatch(img1.des, img2.des, k = 2) 
    good_matches = []

    for m,n in matches:
        if m.distance < ratio * n.distance:
            good_matches.append(m)


    return good_matches

In [ ]:
#   We will store our matches in a dict, with the key corresponding
#   to the indices of the image pair, and the value containing
#   the matched descriptions

match_graph = {}

for i in tqdm(range(len(frames) - 1), desc="Sequential Matching"):
    match_graph[(i, i+1)] = match_pair(frames[i], frames[i+1])

#   We will also do skip matches, where the i-th item is matched with the i+2-th item
#   for robustness

for i in tqdm(range(len(frames) - 2), desc="Skip Matching"):
    match_graph[(i, i+2)] = match_pair(frames[i], frames[i+2])




Skip Matching: 100%|██████████| 100/100 [00:03<00:00, 27.74it/s]


### Extrinsic Matrix

The extrinstic matrix is needed to triangulate points in the 2D images to a 3D space.
To accomplish, we need to a pick a "best" image pair.

In [ ]:
# We define the best pair to be one with the most matches

best_pair = None
max_matches = 0

for (i,j), matches in match_graph.items():
    if len(matches) > max_matches:
        max_matches = len(matches)
        best_pair = (i,j)

print(f"The best pair is {best_pair} with {max_matches} matches")

The best pair is (96, 98) with 1482 matches


We then find the coordinates of each match in both images, and use them to generate the extrinsic matrix.

In [ ]:
imgA = frames[best_pair[0]]
imgB = frames[best_pair[1]]
matches = match_graph[best_pair]

# Gets the coordinates of matches in image A
ptsA = np.float32([imgA.kp[m.queryIdx].pt for m in matches])
# Gets the coordinates of matches in image B
ptsB = np.float32([imgB.kp[m.trainIdx].pt for m in matches])

# Use the RANSAC algorithm for generating extrinsic matrix and non-outlier values (inliers)
extrin, inliers = cv2.findEssentialMat(ptsA, ptsB, intrin, method=cv2.RANSAC, prob=0.999, threshold=1)
inliers = inliers.ravel().astype(bool)

print(f"Number of inliers values: {inliers.sum()}")

Number of inliers values: 1359


### Intial Coordinate Frame and Triangulation

We can now establish an intial coordinate frame based on imgA, and triangulate our match points based on that.

In [ ]:
_, R, t, mask_pose = cv2.recoverPose(extrin, ptsA, ptsB, intrin)

print(f"Recovered R{R}")
print(f"Recovered T{t}")

Recovered R[[ 0.99463336 -0.05156599 -0.08969631]
 [ 0.06258088  0.99022153  0.12467938]
 [ 0.08239    -0.12962355  0.98813442]]
Recovered T[[ 0.74477262]
 [-0.43519288]
 [ 0.50588625]]


In [ ]:
# Only keep inliers points of imgA and imgB, and tranpose them for projection
ptsA_in = ptsA[inliers].T
ptsB_in = ptsB[inliers].T

# Projection matrices based on homogenous coordinates
P1 = intrin @ np.hstack((np.eye(3), np.zeros((3,1))))  # coordinate system based on imgA
P2 = intrin @ np.hstack((R, t))       

# Finds the best pair of 3D dimensional points for our matches 
# based on camera imgA that bestexplains the projection matrices
pts4d = cv2.triangulatePoints(P1, P2, ptsA_in, ptsB_in)

# Convert to inhomogenous coordinates
pts3d = (pts4d[:3] / pts4d[3]).T

# Coordinates of all match points
print(pts3d)


[[-0.33076254 -0.14542705  2.8754756 ]
 [-0.3193271  -0.14610377  2.8682687 ]
 [-0.31092358 -0.14384411  2.8629825 ]
 ...
 [ 0.7899166  -0.19225056  2.8535614 ]
 [ 0.808596   -0.2810078   2.8521192 ]
 [ 0.8461077  -0.23119026  2.8733819 ]]


Now we creating a mapping structure that is able to take each match, find its corresponding xyz points, and then keep track where this point shows up in which images

In [ ]:
map_points = []

# Enumerate through all inlier best-pair matches
for idx, m in enumerate(np.array(match_graph[best_pair])[inliers]):
    # XYZ coords of each match
    coord = pts3d[idx] 
    # Which keypoints observe this point
    obs = {best_pair[0] : m.queryIdx, best_pair[1]: m.trainIdx} 
    # Store this info
    map_points.append({'coord':coord, 'obs':obs})

# Keep track of camera pose information
poses = {best_pair[0] : (np.eye(3), np.zeros((3,1))),
         best_pair[1] : (R,t) }

# Keep track of which match pairs have been registered
registered = set(best_pair)

### Structure from Motion (SfM)

Now that we have an inital set of 3D points relative to imgA, and a registered image consisting of our best pair, we iterate through the other matches in the match_graph, and figure out which the next best image to register next.

In [ ]:
# Finds the next best image to connect to the given the registered image

def next_view(registered, match_graph):

    best_k = None
    best_score = 0

    for (i,j), matches in match_graph.items():

        # We need only one of i or j to be registered
        # If neither are registered, we cannot connect it
        # to the registered image.

        if i in registered and j in registered:
            # Cameras are already registered
            continue
        elif i in registered:
            if len(matches) > best_score:
                best_k = j
                best_score = len(matches)
        elif j in registered:
            if len(matches) > best_score:
                best_k = i
                best_score = len(matches)
        else:
            continue

    return best_k
    

In [ ]:
next_view(registered, match_graph)

99

Once, we have our next best image, we find keypoints in the registered image, and compare it to the keypoints in the best next image. This allow us to correspond 3D points in the map_points, with the 2D points in each image.

In [ ]:
def get_pnp_match(new_k, registered, map_points, frames, match_graph):
    pts3d = []
    pts2d = []

    # Explore all matches involving new_k
    for (i, j), matches in match_graph.items():
        if i in registered and j == new_k:
            for m in matches:
                # Retrieve image coords in both images
                kp_i = m.queryIdx
                kp_k = m.trainIdx

                # search through map_points for a point observed in image i
                for P in map_points:
                    if i in P['obs'] and P['obs'][i] == kp_i:
                        pts3d.append(P['coord'])
                        pts2d.append(frames[new_k].kp[kp_k].pt)

        if j in registered and i == new_k:
            for m in matches:
                kp_j = m.trainIdx
                kp_k = m.queryIdx

                for P in map_points:
                    if j in P['obs'] and P['obs'][j] == kp_j:
                        pts3d.append(P['coord'])
                        pts2d.append(frames[new_k].kp[kp_k].pt)

    return np.array(pts3d, dtype=np.float32), np.array(pts2d, dtype=np.float32)

In [ ]:
# --------------------------
# Incremental SfM Loop (Improved)
# --------------------------
min_pnp_points = 10  # require at least this many 3D-2D correspondences

while len(registered) < len(frames):
    # ---- 1. Choose next image to register ----
    candidate = None
    best_score = 0
    for (i, j), matches in match_graph.items():
        if i in registered and j not in registered:
            # filter matches by PnP correspondences
            pts3d, pts2d = get_pnp_match(j, registered, map_points, frames, match_graph)
            score = len(pts3d)
            if score > best_score:
                candidate = (i, j)
                best_score = score
        elif j in registered and i not in registered:
            pts3d, pts2d = get_pnp_match(i, registered, map_points, frames, match_graph)
            score = len(pts3d)
            if score > best_score:
                candidate = (j, i)
                best_score = score

    if candidate is None or best_score < min_pnp_points:
        print("No more views can be registered.")
        break

    reg_view, new_view = candidate
    print(f"Registering view {new_view} using view {reg_view}")

    # ---- 2. Get 3D–2D correspondences for PnP ----
    pts3d, pts2d = get_pnp_match(new_view, registered, map_points, frames, match_graph)
    if len(pts3d) < min_pnp_points:
        print(f"Not enough PnP points for view {new_view}. Skipping.")
        registered.add(new_view)
        poses[new_view] = poses[reg_view]  # fallback
        continue

    # ---- 3. Solve camera pose via PnP ----
    success, rvec, tvec = cv2.solvePnP(pts3d, pts2d, intrin, None, flags=cv2.SOLVEPNP_ITERATIVE)
    if not success:
        print(f"PnP failed for view {new_view}. Skipping.")
        registered.add(new_view)
        poses[new_view] = poses[reg_view]
        continue

    R, _ = cv2.Rodrigues(rvec)
    t = tvec.reshape(3, 1)
    poses[new_view] = (R, t)
    registered.add(new_view)

    # ---- 4. Triangulate new points ----
    # Get matches for triangulation
    if (reg_view, new_view) in match_graph:
        matches = match_graph[(reg_view, new_view)]
        swapped = False
    elif (new_view, reg_view) in match_graph:
        matches = match_graph[(new_view, reg_view)]
        swapped = True
    else:
        matches = []
        swapped = False

    if len(matches) < 8:
        continue

    ptsA, ptsB = [], []
    for m in matches:
        if not swapped:
            ptsA.append(frames[reg_view].kp[m.queryIdx].pt)
            ptsB.append(frames[new_view].kp[m.trainIdx].pt)
        else:
            ptsA.append(frames[reg_view].kp[m.trainIdx].pt)
            ptsB.append(frames[new_view].kp[m.queryIdx].pt)

    ptsA = np.float32(ptsA).T
    ptsB = np.float32(ptsB).T

    P1 = intrin @ np.hstack([poses[reg_view][0], poses[reg_view][1]])
    P2 = intrin @ np.hstack([poses[new_view][0], poses[new_view][1]])

    pts4d = cv2.triangulatePoints(P1, P2, ptsA, ptsB)
    new_pts3d = (pts4d[:3] / pts4d[3]).T

    # ---- 5. Filter points: behind camera / extreme depth ----
    mask = (new_pts3d[:, 2] > 0) & (new_pts3d[:, 2] < 10)
    new_pts3d = new_pts3d[mask]
    matches = np.array(matches)[mask]

    # ---- 6. Add new points to map_points, merging duplicates ----
    for m, xyz in zip(matches, new_pts3d):
        # check if a nearby point already exists
        if not any(np.linalg.norm(xyz - p['coord']) < 1e-3 for p in map_points):
            if not swapped:
                obs = {reg_view: m.queryIdx, new_view: m.trainIdx}
            else:
                obs = {reg_view: m.trainIdx, new_view: m.queryIdx}
            map_points.append({'coord': xyz, 'obs': obs})

    print(f"Registered view {new_view}, added {len(new_pts3d)} new points, total map points: {len(map_points)}")


Registering view 99 using view 98
Registered view 99, added 1418 new points, total map points: 2638
Registering view 100 using view 99
Registered view 100, added 0 new points, total map points: 2638
Registering view 101 using view 100
Registered view 101, added 0 new points, total map points: 2638
Registering view 97 using view 96
Registered view 97, added 424 new points, total map points: 3040
Registering view 95 using view 96
Registered view 95, added 1002 new points, total map points: 3988
Registering view 94 using view 95
Registered view 94, added 470 new points, total map points: 4433
Registering view 92 using view 94
Registered view 92, added 34 new points, total map points: 4467
Registering view 93 using view 92
Registered view 93, added 323 new points, total map points: 4775
Registering view 91 using view 92
Registered view 91, added 97 new points, total map points: 4871
Registering view 90 using view 91
Registered view 90, added 638 new points, total map points: 5437
Registeri

In [ ]:
import open3d as o3d
points = np.array([p['coord'] for p in map_points])
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.paint_uniform_color([0, 0, 1])  # blue points

# Remove noise
pcd, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
pcd = pcd.voxel_down_sample(voxel_size=0.002)

# Visualize
o3d.visualization.draw_geometries([pcd], window_name="Sparse 3D Reconstruction")

[Open3D WARNING] [ViewControl] SetViewPoint() failed because window height and width are not set.
